# 1. Importación de librerías
Para la comparación empírica de los modelos IBM1 e IBM2, vamos a utilizar las siguientes librerías:
1. Time: Librería que nos permitirá manipular el tiempo en nuestro programa
2. Pandas: Librería de python usada para el análisis y manipulación de datos tabulares y estructurados (filas y columnas).
2. AST: Permite anaízar código de python antes de que se convierta en bytecode.
3. Scikit-learn: Librería 100% de Machine Learning. De esta tomaremos el paquete "*model-selection*" de modo que podamos tener acceso a la función *train_test_split()*
4. NLTK: Librería 100% de PLN. De aquí se extraerán los modelos a comparar, la función "*AlignedSent()*" y la métrica BLEU para comparar los modelos.

In [6]:
import pandas as pd
import ast
import time
from sklearn.model_selection import train_test_split
from nltk.translate import AlignedSent
from nltk.translate.ibm1 import IBMModel1
from nltk.translate.ibm2 import IBMModel2
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.bleu_score import SmoothingFunction

# 2. Carga de datos y separación en "Train y Test"
Para evaluar cualquier modelo de Machine Learning (En el caso de este proyecto, un modelo de traducción), no podemos hacer pruebas con los mismos datos con los que fue entrenado, es por ello que se va a separar el dataset en 2: una parte en "Entrenamiento" y otra en "Pruebas". Idealmente, se suele recomendar una separación 80/20 respectivamente, para esta demostración se utilizará "95/5".

In [2]:
print("2. Preparación de datos")

# 1. Cargar el archivo definitivo
df = pd.read_csv("../data/Corpus_Tokenizado_Final.csv")

# 2. Conversión texto -> listas en python
df['Tokens_Ingles'] = df['Tokens_Ingles'].apply(ast.literal_eval)
df['Tokens_Espanol'] = df['Tokens_Espanol'].apply(ast.literal_eval)

# 3. División de datos (95% entrenamiento, 5% pruebas)
trainData, testData = train_test_split(df, test_size=0.05, random_state=42)
print(f"Cantidad de oraciones para entrenar: {len(trainData)}")
print(f"Cantidad de oraciones para evaluar: {len(testData)}")

# 4. Formatear datos para NLTK
trainingCorpus = []
for es, en in zip(trainData['Tokens_Espanol'], trainData['Tokens_Ingles']):
    trainingCorpus.append(AlignedSent(es, en))

2. Preparación de datos
Cantidad de oraciones para entrenar: 6743
Cantidad de oraciones para evaluar: 355


# 3. Entrenamiento y Tiempos: IBM1 vs IBM2
Aquí se va a poner a competir a ambos modelos. Se entrenarán los dos con las mismas oraciones y mediremos cuántos segundos tarda cada uno en aprender. Esto nos dará un primer argumento: La eficiencia computacional.

In [3]:
print("3. Comparativa de modelos")
iterations = 10

#Entrenamiento de IBM1
print("Entrenando Modelo IBM1...")
start_ibm1 = time.time()
model1 = IBMModel1(trainingCorpus, iterations)
time_ibm1 = time.time() - start_ibm1
print(f"Entrenamiento IBM1 terminado. Tiempo: {time_ibm1}")

#Entrenamiento de IBM2
print("Entrenando Modelo IBM2...")
start_ibm2 = time.time()
model2 = IBMModel2(trainingCorpus, iterations)
time_ibm2 = time.time() - start_ibm2
print(f"Entrenamiento IBM2 terminado. Tiempo: {time_ibm2}")

3. Comparativa de modelos
Entrenando Modelo IBM1...
Entrenamiento IBM1 terminado. Tiempo: 52.893163204193115
Entrenando Modelo IBM2...
Entrenamiento IBM2 terminado. Tiempo: 331.45510506629944


# 4. Traducción y Evaluación con Métrica BLEU
Con el resto del Dataset, se probarán ambos modelos para traducir palabra por palabra cada oración, lo único de lo que se dispondrá es del aprendizaje. Finalmente, aplicaremos la fórmula matemática BLEU para comparar sus traducciones con las traducciones humanas del dataset

In [7]:
print("4. Evaluación BLEU")
chencherry = SmoothingFunction()

def translateSentence(englishTokens, model):
    translation = []
    for enWord in englishTokens:
        #Guardar probabiliadades para esta palabra
        probabilities = model.translation_table[enWord]
        if probabilities:

            #Elegir la palabra en español con la mayor probabilidad
            bestWordSp = max(probabilities, key=probabilities.get)
            if bestWordSp is not None:
                translation.append(bestWordSp)

        #Si no la conoce, se guarda directamente
        else:
            translation.append(enWord)
    return translation

humanRef = []
translationsIBM1 = []
translationsIBM2 = []

print("Traduciendo set de prueba con ambos modelos...")
for index, row in testData.iterrows():
    eng = row['Tokens_Ingles']
    sp_real = row['Tokens_Espanol']

    # BLEU exige que las referencias esten en una lista de listas.
    humanRef.append(sp_real)

    # Traducciones de la máquina
    translationsIBM1.append(translateSentence(eng, model1))
    translationsIBM2.append(translateSentence(eng, model2))

# Cálculo de BLEU
wBleu = (0.5, 0.5, 0, 0)

scoreIBM1 = corpus_bleu(humanRef, translationsIBM1, weights=wBleu, smoothing_function=chencherry.method1)
scoreIBM2 = corpus_bleu(humanRef, translationsIBM2, weights=wBleu, smoothing_function=chencherry.method2)

print("Resultados de métrias BLEU")
print(f"BLEU IBM1: {scoreIBM1}")
print(f"BLEU IBM2: {scoreIBM2}")

4. Evaluación BLEU
Traduciendo set de prueba con ambos modelos...
Resultados de métrias BLEU
BLEU IBM1: 0.0027734742106647753
BLEU IBM2: 0.0031939342949829272
